In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.tokenize import sent_tokenize
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
file_path = '/content/drive/MyDrive/Colab Notebooks/Recommender-System/reviews_Cell_Phones_and_Accessories_5.json/Cell_Phones_and_Accessories_5.json'

In [ ]:
try:
    df = pd.read_json(file_path, lines=True)

    print("\n" + "="*40)
    print("      THÔNG TIN TỔNG QUAN TẬP DỮ LIỆU")
    print("="*40)
    print(f"Tổng số bài đánh giá (Reviews) : {df.shape[0]:,}")
    print(f"Tổng số cột dữ liệu (Features) : {df.shape[1]}")

    print("\n--- KIỂM TRA DỮ LIỆU BỊ THIẾU (NULL/NaN) ---")
    print(df.isnull().sum())

    display(df.head(3))

except FileNotFoundError:
    print("\n[LỖI] Không tìm thấy file! Bạn hãy kiểm tra lại đường dẫn file_path xem đã gõ đúng thư mục trên Drive chưa nhé.")
except Exception as e:
    print(f"\n[LỖI] Có lỗi khác xảy ra: {e}")


      THÔNG TIN TỔNG QUAN TẬP DỮ LIỆU
Tổng số bài đánh giá (Reviews) : 194,439
Tổng số cột dữ liệu (Features) : 9

--- KIỂM TRA DỮ LIỆU BỊ THIẾU (NULL/NaN) ---
reviewerID           0
asin                 0
reviewerName      3519
helpful              0
reviewText           0
overall              0
summary              0
unixReviewTime       0
reviewTime           0
dtype: int64


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A30TL5EWN6DFXT,120401325X,christina,"[0, 0]",They look good and stick good! I just don't li...,4,Looks Good,1400630400,"05 21, 2014"
1,ASY55RVNIL0UD,120401325X,emily l.,"[0, 0]",These stickers work like the review says they ...,5,Really great product.,1389657600,"01 14, 2014"
2,A2TMXE2AFO7ONB,120401325X,Erica,"[0, 0]",These are awesome and make my phone look so st...,5,LOVE LOVE LOVE,1403740800,"06 26, 2014"


In [ ]:
# Tiền xử lí
df_clean = df[['reviewerID', 'asin', 'overall', 'summary', 'reviewText']].copy()
df_clean.fillna('', inplace=True)
df_clean['full_review'] = df_clean['summary'] + " . " + df_clean['reviewText']

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_clean['clean_text'] = df_clean['full_review'].apply(clean_text)
df_clean.drop(columns=['summary', 'reviewText', 'full_review'], inplace=True)

display(df_clean.head(5))

,reviewerID,asin,overall,clean_text
0,A30TL5EWN6DFXT,120401325X,4,looks good they look good and stick good i jus...
1,ASY55RVNIL0UD,120401325X,5,really great product these stickers work like ...
2,A2TMXE2AFO7ONB,120401325X,5,love love love these are awesome and make my p...
3,AWJ0WZQYMYFQ4,120401325X,4,cute item arrived in great time and was in per...
4,ATX7CZYFXI1KW,120401325X,5,leopard home button sticker for iphone 4s awes...


In [ ]:
# #LDA trích xuất khía cạnh
# vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english', max_features=5000)
# dtm = vectorizer.fit_transform(df_clean['clean_text'])

# lda_model = LatentDirichletAllocation(n_components=10, random_state=42)
# lda_model.fit(dtm)

# feature_names = vectorizer.get_feature_names_out()

# for topic_idx, topic in enumerate(lda_model.components_):
#     top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
#     print(f" Aspect {topic_idx + 1}: {', '.join(top_words)}")

In [ ]:
# #Vader (Tính điểm cảm xúc của các aspect)
# nltk.download('vader_lexicon')
# sia = SentimentIntensityAnalyzer()

# dynamic_aspect_dict = {}
# for topic_idx, topic in enumerate(lda_model.components_):
#     aspect_name = f"Aspect_{topic_idx + 1}"
#     top_words =[feature_names[i] for i in topic.argsort()[:-11:-1]]
#     dynamic_aspect_dict[aspect_name] = top_words

# def extract_dynamic_aspect_sentiment(text):
#     sentiment_score = sia.polarity_scores(text)['compound']

#     aspect_scores = {aspect: np.nan for aspect in dynamic_aspect_dict.keys()}

#     words = set(text.split())
#     for aspect, keywords in dynamic_aspect_dict.items():
#         if any(kw in words for kw in keywords):
#             aspect_scores[aspect] = sentiment_score

#     return pd.Series(aspect_scores)

# df_aspects = df_clean['clean_text'].apply(extract_dynamic_aspect_sentiment)
# df_final = pd.concat([df_clean[['reviewerID', 'asin', 'overall', 'clean_text']], df_aspects], axis=1)

# display(df_final.head(5))

In [ ]:
# nltk.download('vader_lexicon', quiet=True)
# nltk.download('punkt', quiet=True)
# nltk.download('punkt_tab', quiet=True)
# sia = SentimentIntensityAnalyzer()

# # 2. TỪ ĐIỂN KHÍA CẠNH ĐÃ TINH CHỈNH (Human-in-the-loop)
# # Được gộp và chắt lọc từ chính 10 Topic của LDA, bỏ đi các từ rác (phone, like, good...)
# final_aspect_dict = {
#     'Screen': {'screen', 'protector', 'bubbles', 'dust', 'clear'},
#     'Design': {'color', 'cute', 'looks', 'nice', 'style', 'shape'},
#     'Protection': {'case', 'protection', 'fit', 'cover', 'drop'},
#     'Price_Quality': {'price', 'quality', 'cheap', 'worth', 'value', 'sturdy'},
#     'Power_Charging': {'charge', 'battery', 'power', 'usb', 'cable', 'life', 'charger'},
#     'Audio': {'sound', 'bluetooth', 'headset', 'ear', 'music', 'speaker'}
# }

# print("Đang tiến hành chấm điểm Cảm xúc cho 6 Khía cạnh cốt lõi (Bỏ qua từ rác)...")

# def extract_final_sentiment(text):
#     # Đảm bảo text là chuỗi hợp lệ
#     if not isinstance(text, str):
#         return pd.Series({aspect: np.nan for aspect in final_aspect_dict.keys()})

#     sentences = sent_tokenize(text)

#     aspect_score_lists = {aspect: [] for aspect in final_aspect_dict.keys()}

#     for sentence in sentences:
#         sentence_sentiment = sia.polarity_scores(sentence)['compound']

#         clean_sentence = re.sub(r'[^a-z0-9\s]', ' ', sentence.lower())
#         words = set(clean_sentence.split())

#         for aspect, keywords in final_aspect_dict.items():
#             if keywords & words:
#                 aspect_score_lists[aspect].append(sentence_sentiment)

#     final_scores = {}
#     for aspect, scores in aspect_score_lists.items():
#         if len(scores) > 0:
#             final_scores[aspect] = np.mean(scores)
#         else:
#             final_scores[aspect] = np.nan

#     return pd.Series(final_scores)

# df_clean['full_review_with_punct'] = df['summary'].fillna('') + ". " + df['reviewText'].fillna('')

# df_aspects = df_clean['full_review_with_punct'].apply(extract_final_sentiment)

# df_final = pd.concat([df_clean[['reviewerID', 'asin', 'overall', 'clean_text']], df_aspects], axis=1)
# display(df_final.head(10))


In [ ]:
# Cài đặt Unsloth và các thư viện hỗ trợ để chạy LLM 4-bit trên Colab T4
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", # Bản 4-bit cực nhẹ
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model) # Tối ưu hóa cho việc dự đoán

In [ ]:
import os
import json
import re
import pandas as pd
from google.colab import drive

# 1. Khởi tạo đường dẫn (Drive của bạn)
drive.mount('/content/drive')
save_dir = '/content/drive/MyDrive/Colab Notebooks/Recommender-System'
os.makedirs(save_dir, exist_ok=True)
save_path = f'{save_dir}/aspect.csv'

# 2. Hàm trích xuất bằng Qwen2.5 (Giữ nguyên của bạn)
def extract_aspects_with_qwen(text):
    prompt = f"""<|im_start|>system
Bạn là chuyên gia phân tích đánh giá sản phẩm. Hãy trích xuất điểm cảm xúc từ -1.0 đến 1.0 cho 6 khía cạnh: Screen, Design, Protection, Price_Quality, Power_Charging, Audio.
Chỉ trả về duy nhất định dạng JSON. Nếu không nhắc tới hãy để null.<|im_end|>
<|im_start|>user
Review: "{text}"<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 128, temperature = 0.1)
    response = tokenizer.batch_decode(outputs)[0]

    try:
        json_str = re.search(r'\{.*\}', response, re.DOTALL).group()
        return json.loads(json_str)
    except:
        return {a: None for a in ['Screen', 'Design', 'Protection', 'Price_Quality', 'Power_Charging', 'Audio']}

# 3. KIỂM TRA ĐIỂM LƯU (CHECKPOINT)
if os.path.exists(save_path):
    df_done = pd.read_csv(save_path)
    start_index = len(df_done)
    print(f"♻️ Tìm thấy file aspect.csv! Tiếp tục chạy từ dòng thứ {start_index}...")
else:
    df_done = pd.DataFrame()
    start_index = 0
    print("🚀 Bắt đầu phân tích mới hoàn toàn và sẽ lưu vào aspect.csv...")

# 4. CẤU HÌNH CHẠY THEO LÔ
TOTAL_TO_RUN = 5000  # Bạn có thể đổi số này (VD: 5000 dòng để làm báo cáo)
BATCH_SIZE = 50      # Cứ chạy xong 50 dòng thì lưu 1 lần (Vì LLM ngốn RAM, để 50 cho an toàn tuyệt đối)
aspect_cols = ['Screen', 'Design', 'Protection', 'Price_Quality', 'Power_Charging', 'Audio']

print("-" * 50)

# 5. VÒNG LẶP XỬ LÝ AN TOÀN
for i in range(start_index, TOTAL_TO_RUN, BATCH_SIZE):
    batch = df_clean.iloc[i : i + BATCH_SIZE].copy()
    print(f"⏳ Đang xử lý từ dòng {i} đến {i + len(batch)}...")

    # Cho Qwen đọc và phân tích
    results = batch['clean_text'].apply(extract_aspects_with_qwen)
    df_llm_aspects = pd.DataFrame(results.tolist(), index=batch.index)

    # THUẬT TOÁN LÀM SẠCH (CỰC KỲ QUAN TRỌNG):
    # Ép chữ thành Số thực (Float). Biến 'null', 'NaN' thành số 0.0 để tính toán ma trận
    df_llm_aspects[aspect_cols] = df_llm_aspects[aspect_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)

    # Ghép với các cột gốc
    batch_final = pd.concat([batch[['reviewerID', 'asin', 'overall', 'clean_text']], df_llm_aspects], axis=1)

    # Nối vào dữ liệu tổng và Ghi đè vào Drive
    df_done = pd.concat([df_done, batch_final], ignore_index=True)
    df_done.to_csv(save_path, index=False)

    print(f"✅ Đã lưu thành công lô {i} -> {i + len(batch)} vào Drive!")

print("🎉 XUẤT SẮC! Đã chạy xong toàn bộ dữ liệu chỉ định.")

In [ ]:
# ==========================================
# PHASE 2: Profiling phase => xây dựng User Profile và Item Profile
# ==========================================
# 1. Khai báo danh sách 6 khía cạnh (đảm bảo trùng với Phase 1)
aspect_columns = ['Screen', 'Design', 'Protection', 'Price_Quality', 'Power_Charging', 'Audio']

# 2. Đọc dữ liệu từ file aspect.csv đã lưu ở Phase 1 lên RAM
# Biến save_path là đường dẫn, df_final sẽ là bảng dữ liệu thực tế
if os.path.exists(save_path):
    df_final = pd.read_csv(save_path)
    print(f"Đã tải dữ liệu từ {save_path}")
else:
    print("Không tìm thấy file aspect.csv. Hãy kiểm tra lại đường dẫn!")

In [ ]:
# 3. XÂY DỰNG ITEM PROFILES (Hồ sơ Sản phẩm)
# Gọi groupby trên bảng df_final (không phải trên biến save_path)
item_profiles = df_final.groupby('asin')[aspect_columns].mean()
item_profiles = item_profiles.fillna(0.0)
display(item_profiles.head())

In [ ]:
# ==========================================
# 4. XÂY DỰNG USER PROFILES (Hồ sơ Người dùng)
# ==========================================
user_profiles = df_final.groupby('reviewerID')[aspect_columns].mean()
user_profiles = user_profiles.fillna(0.0)

display(user_profiles.head())

In [ ]:
# ==========================================
# 5. LƯU CÁC FILE .PKL ĐỂ DEPLOY
# ==========================================
# Sử dụng save_dir đã định nghĩa ở cell trước
item_profiles.to_pickle(f'{save_dir}/item_profiles.pkl')
user_profiles.to_pickle(f'{save_dir}/user_profiles.pkl')
print(f"\n Đã lưu file 'item_profiles.pkl' và 'user_profiles.pkl' thành công!")

In [ ]:
print("=======================================================")
print(" PHASE 3: HYBRID RECOMMENDATION (CHỈ TRAIN TRÊN 80% DATA)")
print("=======================================================")

# [MỚI] CHIA DỮ LIỆU ĐỂ TRÁNH RÒ RỈ (DATA LEAKAGE)
print("[1/5] Chia tập dữ liệu Train/Test (80/20)...")
df_train, df_test = train_test_split(df_final, test_size=0.2, random_state=42)
print(f"      -> Số lượng đánh giá: Train={len(df_train)}, Test={len(df_test)}")

# ---------------------------------------------------------
# Bước 3.1: Huấn luyện TruncatedSVD cho Collaborative Filtering
# ---------------------------------------------------------
print("[2/5] Chuẩn bị ma trận User-Item cho SVD (chỉ dùng tập Train)...")

# Mapping user/item sang index số (Vẫn giữ nguyên bộ khung kích thước cho toàn bộ User/Item)
user_enc = {u: i for i, u in enumerate(user_profiles.index)}
item_enc = {i: j for j, i in enumerate(item_profiles.index)}

num_users = len(user_profiles)
num_items = len(item_profiles)

# Tạo ma trận sparse User x Item (CHỈ LẤY ĐIỂM TỪ df_train)
ratings_matrix_train = np.zeros((num_users, num_items))
for idx, row in df_train.iterrows():
    u_idx = user_enc[row['reviewerID']]
    i_idx = item_enc[row['asin']]
    ratings_matrix_train[u_idx, i_idx] = row['overall']

print("      -> Ma trận User-Item (Train) đã sẵn sàng.")

# Huấn luyện TruncatedSVD
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(ratings_matrix_train)  # Chỉ học từ tập Train
item_factors = svd.components_.T

print("      -> Huấn luyện TruncatedSVD hoàn tất!")

# Bước 3.2: Chuẩn bị ma trận Aspect similarity
# ---------------------------------------------------------
all_items = item_profiles.index.tolist()
item_matrix = item_profiles.values
print("[3/5] Ma trận Aspect similarity đã sẵn sàng.")

# ---------------------------------------------------------
# Bước 3.3: Hàm Gợi ý Hybrid
# ---------------------------------------------------------
def get_hybrid_recommendations(user_id, alpha=0.6, top_n=5):
    if user_id not in user_profiles.index:
        return "Lỗi: User chưa có profile trong hệ thống (Cold-start)."

    u_idx = user_enc[user_id]
    user_vector = user_factors[u_idx].reshape(1, -1)

    svd_scores = np.dot(user_vector, item_factors.T).flatten()
    # Tránh chia cho 0 nếu svd_scores max = min
    if svd_scores.max() == svd_scores.min():
        svd_scores_norm = np.zeros_like(svd_scores)
    else:
        svd_scores_norm = (svd_scores - svd_scores.min()) / (svd_scores.max() - svd_scores.min())

    u_aspect_vector = user_profiles.loc[user_id].values.reshape(1, -1)
    aspect_sim = cosine_similarity(u_aspect_vector, item_matrix).flatten()
    aspect_sim_norm = (aspect_sim + 1) / 2

    hybrid_score = alpha * svd_scores_norm + (1 - alpha) * aspect_sim_norm

    rec_df = pd.DataFrame({
        'asin': all_items,
        'svd_score': svd_scores_norm,
        'aspect_similarity': aspect_sim_norm,
        'hybrid_score': hybrid_score
    })

    rec_df = rec_df.sort_values(by='hybrid_score', ascending=False).head(top_n)
    return rec_df

# ---------------------------------------------------------
# Bước 3.4: Lưu mô hình & Profile & Tập Test
# ---------------------------------------------------------
print("[4/5] Lưu mô hình, profile và tập Test...")
save_dir = '/content/drive/MyDrive/Colab Notebooks/Recommender-System/models'
os.makedirs(save_dir, exist_ok=True)

with open(os.path.join(save_dir, 'svd_model.pkl'), 'wb') as f:
    pickle.dump({'svd': svd, 'user_factors': user_factors, 'item_factors': item_factors,
                 'user_enc': user_enc, 'item_enc': item_enc}, f)

user_profiles.to_pickle(os.path.join(save_dir, 'user_profiles.pkl'))
item_profiles.to_pickle(os.path.join(save_dir, 'item_profiles.pkl'))

# Lưu lại df_test để dùng cho cell Đánh giá
df_test.to_pickle(os.path.join(save_dir, 'df_test.pkl'))

print(f"      -> Lưu hoàn tất tại: {save_dir}")
print("[5/5] PHASE 3 HOÀN TẤT. SẴN SÀNG ĐÁNH GIÁ (EVALUATION).")

In [ ]:
print("=======================================================")
print(" PHASE 3.5: FINE-TUNING TRỌNG SỐ & ĐÁNH GIÁ MÔ HÌNH")
print("=======================================================")
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity

# 1. Chuẩn bị ma trận "Đáp án" từ tập Test (20%)
ratings_matrix_true = np.zeros((num_users, num_items))
for idx, row in df_test.iterrows():
    u_idx = user_enc[row['reviewerID']]
    i_idx = item_enc[row['asin']]
    ratings_matrix_true[u_idx, i_idx] = row['overall']

# Lấy các vị trí mà User có đánh giá thật trong tập Test (mask)
mask = ratings_matrix_true > 0
y_true = ratings_matrix_true[mask]

# Hàm đánh giá Top-K
def evaluate_ranking(hybrid_matrix, ratings_matrix_true, k=5, threshold=4):
    precisions, recalls, ndcgs = [], [], []
    for u_idx in range(hybrid_matrix.shape[0]):
        pred_scores = hybrid_matrix[u_idx]
        true_ratings = ratings_matrix_true[u_idx]
        relevant_idx = np.where(true_ratings >= threshold)[0]
        if len(relevant_idx) == 0: continue

        top_k_idx = np.argsort(pred_scores)[::-1][:k]
        hits = len(set(top_k_idx) & set(relevant_idx))
        precisions.append(hits / k)
        recalls.append(hits / len(relevant_idx))

        gains = true_ratings[top_k_idx]
        discounts = np.log2(np.arange(2, k + 2))
        dcg = np.sum(gains / discounts)
        ideal_idx = np.argsort(true_ratings)[::-1][:k]
        idcg = np.sum(true_ratings[ideal_idx] / discounts)
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

# 2. Chạy thử nghiệm (Grid Search) để tìm Alpha tối ưu
# Alpha = 0.0: Chỉ dùng Content-based (Aspect)
# Alpha = 1.0: Chỉ dùng Collaborative Filtering (SVD)
alphas_to_test = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

best_alpha = 0
best_ndcg = 0

print("Đang chạy vòng lặp tìm Alpha tốt nhất...")
print(f"{'Alpha':<10} | {'RMSE':<10} | {'MAE':<10} | {'Prec@5':<10} | {'Rec@5':<10} | {'NDCG@5':<10}")
print("-" * 75)

for alpha in alphas_to_test:
    hybrid_matrix = np.zeros((num_users, num_items))
    for u_idx, user_id in enumerate(user_profiles.index):
        u_vector = user_factors[u_idx].reshape(1, -1)
        svd_scores = np.dot(u_vector, item_factors.T).flatten()
        if svd_scores.max() != svd_scores.min():
            svd_scores_norm = (svd_scores - svd_scores.min()) / (svd_scores.max() - svd_scores.min())
        else:
            svd_scores_norm = np.zeros_like(svd_scores)

        u_aspect_vector = user_profiles.loc[user_id].values.reshape(1, -1)
        aspect_sim = cosine_similarity(u_aspect_vector, item_matrix).flatten()
        aspect_sim_norm = (aspect_sim + 1) / 2

        hybrid_matrix[u_idx] = alpha * svd_scores_norm + (1 - alpha) * aspect_sim_norm

    # FIX LỖI ĐIỂM: Ánh xạ [0, 1] về thang điểm [1, 5] thay vì [0, 5]
    y_pred = hybrid_matrix[mask] * 4 + 1

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    prec, rec, ndcg = evaluate_ranking(hybrid_matrix, ratings_matrix_true, k=5)

    print(f"{alpha:<10.1f} | {rmse:<10.4f} | {mae:<10.4f} | {prec:<10.4f} | {rec:<10.4f} | {ndcg:<10.4f}")

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_alpha = alpha

print("-" * 75)
print(f"🌟 ALPHA TỐT NHẤT CHO MÔ HÌNH LÀ: {best_alpha}")

In [ ]:
!pip install lime

In [ ]:
import pickle, os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

try:
    import lime
    import lime.lime_tabular
except ImportError:
    print("Vui lòng chạy lệnh: !pip install lime")

# 1. NẠP MÔ HÌNH VÀ DỮ LIỆU
save_dir = '/content/drive/MyDrive/Colab Notebooks/Recommender-System/models'
with open(os.path.join(save_dir, 'svd_model.pkl'), 'rb') as f:
    m = pickle.load(f)
    svd, user_factors, item_factors, user_enc, item_enc = m['svd'], m['user_factors'], m['item_factors'], m['user_enc'], m['item_enc']

user_profiles = pd.read_pickle(os.path.join(save_dir, 'user_profiles.pkl'))
item_profiles = pd.read_pickle(os.path.join(save_dir, 'item_profiles.pkl'))
item_matrix, aspect_names = item_profiles.values, item_profiles.columns.tolist()
print("✅ Đã load mô hình và profile thành công!")

# 2. HÀM DỰ BÁO HYBRID (Đã fix lỗi chia cho 0)
def predict_hybrid(user_id, alpha=0.5, top_n=5):
    if user_id not in user_enc: return pd.DataFrame()

    u_vec = user_factors[user_enc[user_id]].reshape(1, -1)
    svd_scores = np.dot(u_vec, item_factors.T).flatten()

    # [FIX LỖI 1]: Chống chia cho 0
    if svd_scores.max() == svd_scores.min():
        svd_norm = np.zeros_like(svd_scores)
    else:
        svd_norm = (svd_scores - svd_scores.min()) / (svd_scores.max() - svd_scores.min())

    asp_sim = cosine_similarity(user_profiles.loc[user_id].values.reshape(1, -1), item_matrix).flatten()
    asp_norm = (asp_sim + 1) / 2

    scores = alpha * svd_norm + (1 - alpha) * asp_norm
    return pd.DataFrame({'asin': item_profiles.index, 'score': scores}).sort_values('score', ascending=False).head(top_n)

# 3. HÀM GIẢI THÍCH LIME (Đã fix lỗi sụp đổ phân phối ngẫu nhiên)
def explain_with_lime(user_id, item_asin, alpha=0.5):
    print(f"\n🔍 ĐANG TẠO BẢN GIẢI THÍCH LIME CHO SẢN PHẨM {item_asin} ...")

    u_idx, u_asp = user_enc[user_id], user_profiles.loc[user_id].values.reshape(1, -1)
    svd_scores = np.dot(user_factors[u_idx].reshape(1, -1), item_factors.T).flatten()

    # [FIX LỖI 1]: Chống chia cho 0
    if svd_scores.max() == svd_scores.min():
        svd_norm = np.zeros_like(svd_scores)
    else:
        svd_norm = (svd_scores - svd_scores.min()) / (svd_scores.max() - svd_scores.min())

    data_feats = np.column_stack([svd_norm, item_matrix])

    # [FIX LỖI 2]: Bơm nhiễu siêu vi (Micro-noise) để độ lệch chuẩn luôn > 0
    noise = np.random.normal(0, 1e-5, data_feats.shape)
    data_feats_noisy = data_feats + noise

    feat_names = ['Hành_vi_đám_đông_SVD'] + aspect_names

    # Khởi tạo LIME với dữ liệu đã thêm nhiễu
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=data_feats_noisy, feature_names=feat_names,
        class_names=['Hybrid_Score'], mode='regression', random_state=42
    )

    def lime_predict(p_data):
        sims_norm = (cosine_similarity(p_data[:, 1:], u_asp).flatten() + 1) / 2
        return alpha * p_data[:, 0] + (1 - alpha) * sims_norm

    item_idx = np.where(item_profiles.index == item_asin)[0][0]
    exp = explainer.explain_instance(data_feats_noisy[item_idx], lime_predict, num_features=4)
    exp.show_in_notebook(show_table=True, show_all=False)

# Dịch kết quả LIME thành văn bản
    print(f"\n💡 LÝ DO HỆ THỐNG CHỌN SẢN PHẨM '{item_asin}':")
    print("-" * 55)
    for cond, weight in exp.as_list():
        # [SỬA LỖI Ở ĐÂY]: Tìm tên đặc điểm chính xác nằm trong chuỗi LIME
        feat = "Không xác định"
        for name in feat_names:
            if name in cond:
                feat = name
                break

        if weight > 0:
            msg = "Đám đông có gu giống bạn cũng thích SP này." if "SVD" in feat else f"Đặc điểm '{feat}' rất khớp với sở thích của bạn."
            print(f"✅ CỘNG (+{weight:.4f}) : {msg}")
        else:
            msg = "SP này ít được nhóm người giống bạn mua." if "SVD" in feat else f"Đặc điểm '{feat}' chưa thực sự hoàn hảo với bạn."
            print(f"❌ TRỪ  ({weight:.4f}) : {msg}")

# 4. CHẠY THỬ NGHIỆM
TEST_USER = "A30TL5EWN6DFXT"
TEST_ALPHA = 0.5

print(f"--- TOP 5 GỢI Ý CHO USER {TEST_USER} ---")
top5 = predict_hybrid(TEST_USER, alpha=TEST_ALPHA)
display(top5)

if not top5.empty:
    explain_with_lime(TEST_USER, top5.iloc[0]['asin'], alpha=TEST_ALPHA)